# 聚类分析

## 课程：金融数据分析与建模

在前几章中，每个样本都带有标签：违约或不违约、涨或跌。这一章转入**无监督学习**的第二个核心工具——聚类分析（Cluster Analysis）。聚类的目标是：在没有任何标签的情况下，仅凭样本本身的特征相似度，自动发现数据中的自然分组结构。

金融领域对聚类的需求无处不在：

- **客户分群**：将客户按消费行为、风险偏好归入不同群体，支撑差异化产品设计
- **市场状态识别**：将历史时间段归类为牛市、熊市、震荡市等状态
- **债券/股票分类**：根据风险特征将证券归入不同类别，辅助组合构建
- **异常检测**：识别不属于任何正常簇的离群点（潜在欺诈或信用风险）

本章介绍两类最经典的聚类方法：

$$
\underbrace{\text{K-means}}_{\text{划分型：先定 K，迭代优化}}
\qquad\text{vs}\qquad
\underbrace{\text{层次聚类}}_{\text{凝聚型：不定 K，树状合并}}
$$

---

**本章使用的数据集**

| 数据集 | 变量 | 说明 |
|---|---|---|
| 数据集 C（客户）| 年均消费额 × 消费频率（500 名客户）| K-means 主例，3 个规则球形簇 |
| 数据集 B（债券）| 久期、信用利差、流动性溢价、评级得分（200 支债券）| 层次聚类主例，4 个簇 |
| 数据集 N（非球形）| 月牙形、同心圆、细长椭圆（300 个点）| K-means 局限演示 |

---

## 从一个分群问题出发

图 1 展示了 500 名客户的年均消费额和月均消费频率。数据没有任何标签——我们不知道哪个客户属于哪类。

![客户数据集](./figs/ml_cluster_fig01_data_C.png){width=80%}

**图 1** 500 名客户的消费行为散点图（未聚类）。直觉上，点云似乎分成了若干组，但边界在哪里、应该分成几组，并不显然。聚类算法的任务，就是将这种直觉数量化。

从图 1 中，大部分人凭直觉会说「有 3 组」。但如果数据是 10 维的，我们无法画散点图，直觉就完全失效了。我们需要一个有明确数学定义、可以推广到任意维度的分组标准。

---

## 两个核心概念：簇内距离与簇间距离

在正式介绍算法之前，先明确「好的聚类」意味着什么。

评价一个聚类方案的好坏，需要同时看两个方面：

- **簇内距离（Intracluster Distance）**：同一个簇内部样本之间的距离。  越**小**越好——说明同簇的点紧密聚集，内部一致性高。

- **簇间距离（Intercluster Distance）**：不同簇之间样本的距离。  越**大**越好——说明不同簇之间差异明显，边界清晰。

![簇内距离与簇间距离](./figs/ml_cluster_fig00a_intra_inter.png){width=90%}

**图 0** 好的聚类满足两个条件：簇内距离小（蓝色双向箭头短，同簇点紧凑）；簇间距离大（红色双向箭头长，两簇分离）。K-means 的目标函数就是对这一直觉的数学表达。

::: {.callout-note}
### 距离度量：如何衡量两点之间的远近

聚类算法依赖对「距离」的定义。最常用的三种距离度量为：

**欧氏距离（Euclidean Distance）**——K-means 的默认距离

$$
\|A - B\| = \sqrt{(x_1-x_2)^2 + (y_1-y_2)^2}
$$

衡量两点之间的直线距离，适合各向同性、量纲相同的数据。

**曼哈顿距离（Manhattan Distance）**

$$
\|A - B\| = |x_1-x_2| + |y_1-y_2|
$$

沿坐标轴的折线距离，对极端异常值比欧氏距离更鲁棒。

**切比雪夫距离（Chebyshev Distance）**

$$
\|A - B\| = \max\{|x_1-x_2|,\, |y_1-y_2|\}
$$

取各维度差值的最大值，用于需要最坏情况度量的场景（较少使用）。

**实践注意**：sklearn 的 `KMeans` 固定使用欧氏距离，无法更换。如果需要曼哈顿距离，可改用 `sklearn.cluster.AgglomerativeClustering`（层次聚类支持自定义距离矩阵）。
:::

---

## K-means 聚类

### 目标：最小化簇内方差

K-means 将簇内距离的思想数量化：将 $n$ 个样本划分为 $K$ 个不重叠的簇 $C_1, C_2, \ldots, C_K$，使每个簇内部尽可能紧凑。数学上，最小化**簇内方差之和**（Within-Cluster Sum of Squares，WCSS，也称 inertia）：

$$
\min_{C_1,\ldots,C_K} \sum_{k=1}^K \sum_{i \in C_k}
\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2 \tag{1}
$$

其中 $\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{i \in C_k} \mathbf{x}_i$是第 $k$ 个簇的**中心**（质心）。

直觉上：我们希望每个簇内部的点都尽量靠近自己的中心，而不同簇的中心彼此尽量远离。这正是同时最小化簇内距离、最大化簇间距离的含义。

### 两步迭代算法

遍历所有可能的划分方案来精确求解公式 (1) 是 NP-hard 问题。K-means 采用简单高效的**两步迭代**近似算法：

1. **初始化**：随机选取 $K$ 个点作为初始簇中心
2. **重复以下两步直到收敛**：
   - **分配步（E 步）**：将每个样本分配到距离最近的簇中心
   $$c_i = \arg\min_k \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2 \tag{2}$$
   - **更新步（M 步）**：将每个簇中心更新为当前成员的均值
   $$\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{i: c_i=k} \mathbf{x}_i \tag{3}$$
3. **收敛条件**：当簇分配不再发生变化时停止

每次迭代都严格减小目标函数 (1)，因此算法一定会收敛（附录 A）。

图 2 展示了 K-means 在客户数据集上的前 4 步迭代过程。

![K-means迭代](./figs/ml_cluster_fig02_kmeans_iter.png){width=100%}

**图 2** K-means 迭代过程（K=3）。步骤 1：随机放置 3 个初始中心（★）；步骤 2：每个点分配给最近的中心，颜色代表所属簇；步骤 3：箭头显示中心从旧位置（空心星）移向新位置（实心星）；步骤 4：再次分配后算法收敛。

> 💡 **动态演示**：K-means 的迭代过程用动图更直观，可参考 [KNIME 可视化](https://www.knime.com/blog/what-is-clustering-how-does-it-work)（HTML 版本可直接播放 GIF）。

收敛后的聚类结果如图 3 所示。

![K-means最终结果](./figs/ml_cluster_fig03_kmeans_result.png){width=80%}

**图 3** K-means 聚类结果（K=3）。背景色为 Voronoi 图——每个区域内的点被分配到对应颜色的中心。三个簇分别对应「价格敏感型」（低消费低频）、「大额偶发型」（高消费低频）和「日常高频型」（中消费高频）客户。

### K 值对 Voronoi 分割的影响

K 的选择直接决定了特征空间被分割成几个区域。图 4a 展示了同一数据集在 K=2, 3, 4, 6 时的 Voronoi 分割效果。

![不同K的Voronoi](./figs/ml_cluster_fig04a_voronoi_k.png){width=100%}

**图 4a** 不同 K 值对 Voronoi 分割区域的影响。K 越大，WCSS 越小（分割越细），但过多的簇会失去实际意义——K=6 时每个大簇被进一步切碎，与真实的 3 组结构不符。这正是「K 越大 WCSS 越低，但不代表越好」的直觉来源。

### 局部最优：初始化的影响

K-means 最重要的实践注意事项是：**结果依赖于初始化，可能陷入局部最优**。

图 4 展示了三种不同初始化的结果。

![局部最优](./figs/ml_cluster_fig04_local_optima.png){width=100%}

**图 4** 不同初始化的 K-means 结果。(a)(b) 初始中心位置不好，算法陷入局部最优，WCSS 较大；(c) K-means++ 初始化，选到了接近全局最优的起点，WCSS 最小。

**解决方案**：sklearn 默认使用 **K-means++** 初始化——第一个中心随机选取，后续每个中心以正比于到已有中心距离平方的概率被选中，迫使初始中心尽可能分散。此外设置 `n_init=20` 重复运行多次，取 WCSS 最小的结果。

```python
km = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42)
```

### K-means 对离群值的敏感性

K-means 的另一个重要局限是**对离群值极度敏感**。由于簇中心是所有成员的均值，一个极端离群点会将整个中心从真实位置「拉偏」。

图 4b 对比了有无离群点时的聚类结果。

![离群值影响](./figs/ml_cluster_fig04b_outlier.png){width=90%}

**图 4b** K-means 对离群值的敏感性。(a) 无离群点时，三个簇中心准确定位在各簇的真实中心；(b) 加入 2 个极端离群点（红色 ×）后，箭头显示中心被明显拉偏，原本清晰的三簇结构受到干扰。

**应对策略**：在聚类之前，应用异常值检测（如 IQR 规则或 Isolation Forest）识别并处理极端离群点；或改用对离群值更鲁棒的 K-medoids 算法（以实际样本点而非均值作为中心）。

---

## 如何选择 K？

K-means 需要预先指定簇的数量 $K$，而这在真实问题中往往是未知的。下面介绍两种最常用的方法。

### 肘部法则（Elbow Method）

随着 $K$ 增大，WCSS 必然单调下降——极端情况下 $K=n$ 时每个点自成一簇，WCSS=0。**肘部法则**寻找 WCSS 曲线斜率急剧减缓的「肘部」，该点之后增加簇数带来的收益开始大幅递减。

![肘部法则](./figs/ml_cluster_fig05_elbow.png){width=80%}

**图 5** 客户数据集的肘部法则。K=3 之前 WCSS 快速下降，K=3 之后趋于平缓，肘部清晰指向 K=3。

### 轮廓系数（Silhouette Score）

肘部法则有时肘部不明显。**轮廓系数**是一个综合考虑簇内紧凑性和簇间分离度的指标。

对第 $i$ 个样本，设：
- $a_i$：该样本到**同簇所有其他样本**的平均距离（越小越好）
- $b_i$：该样本到**最近的其他簇**所有样本的平均距离（越大越好）

则第 $i$ 个样本的轮廓系数为：

$$
s_i = \frac{b_i - a_i}{\max(a_i,\, b_i)} \in [-1, 1] \tag{4}
$$

图 6a 给出了轮廓系数的几何直觉。

![轮廓系数几何直觉](./figs/ml_cluster_fig06a_sil_geometry.png){width=90%}

**图 6a** 轮廓系数的几何含义。$a_i$（蓝色箭头）是目标点到同簇其他点的平均距离，体现簇内凝聚度；$b_i$（红色箭头）是目标点到最近的其他簇所有点的平均距离，体现簇间分离度。当 $b_i \gg a_i$ 时，$s_i \to 1$，说明该点分类理想。

::: {.callout-note}
### 轮廓系数的取值含义

| $s_i$ 的值 | 含义 |
|:---:|---|
| $s_i \approx 1$ | 样本离自己的簇很近，离其他簇很远，分类理想 |
| $s_i \approx 0$ | 样本处于两个簇的边界上，归属模糊 |
| $s_i < 0$ | 样本距离别的簇比本簇更近，可能被分到了错误的簇 |

所有样本的轮廓系数均值即为整个聚类方案的**平均轮廓系数**，越高越好。
在 sklearn 中：
```python
from sklearn.metrics import silhouette_score
score = silhouette_score(X_sc, labels)   # 返回平均轮廓系数
```
:::

![轮廓系数图](./figs/ml_cluster_fig06_silhouette.png){width=100%}

**图 6** (a) 不同 K 对应的平均轮廓系数，K=3 时最高（橙色柱）；(b) K=3 时每个样本的逐点轮廓系数图——三个色块分别对应三个簇，宽度代表该簇样本数，色块均匀且无负值说明聚类效果好。

两种方法在本例中都一致指向 K=3，结论稳健。在真实数据中，建议**同时使用两种方法**，只在二者一致时才有信心确定 K 的取值。

::: {.callout-tip}
### 选 K 没有唯一正确答案

肘部法则和轮廓系数只是辅助判断工具，而非裁判。

在实践中，最终 K 的选择还需要结合**业务逻辑**：
分 3 组还是分 5 组，哪个对后续的产品设计或风险管理更有意义？一个从业务出发说得通、轮廓系数略低一点的方案，通常比一个轮廓系数最高但无法解释的方案更有价值。
:::

---

## 层次聚类

### 自底向上的思路

层次聚类（Hierarchical Clustering）的思路与 K-means 完全不同：

- 无需预先指定 $K$
- 从「每个点自成一簇」开始，不断合并最相似的两个簇，直到所有点合并为一簇
- 全过程记录在**树状图**（dendrogram）中，用户事后再决定切割位置

**凝聚型层次聚类**的步骤：

1. 初始化：$n$ 个簇，每个样本自成一簇
2. 计算所有簇对之间的距离
3. 合并距离最小的两个簇
4. 更新距离矩阵
5. 重复步骤 3–4，直到只剩一个簇

时间复杂度为 $O(n^2 \log n)$，对中等规模数据（$n \leq 10^4$）完全可行。

### 树状图：聚类过程的完整记录

树状图的纵轴是**合并距离**（越高说明合并时两簇差异越大），横轴是样本。图 7 展示了债券数据集的树状图。

![层次聚类树状图](./figs/ml_cluster_fig07_dendrogram.png){width=100%}

**图 7** 200 支债券的层次聚类树状图（Ward 连接）。树状图从底部（每个点自成一簇）向上生长，每次合并对应图中的一个倒 U 形节点，节点高度表示合并时的距离。橙色虚线是切割线——在这个高度切割可以得到 4 个簇，与 4 类债券（国债、投资级企业债、高收益债、新兴市场债）对应。

### 四种连接方式（Linkage）

层次聚类中，两个簇之间的距离可以用多种方式定义，称为**连接方式**（Linkage）。

| 连接方式 | 两簇距离定义 | 特点 |
|---|---|---|
| **单连接**（Single） | 两簇中最近的两点距离 | 容易形成链式结构，对噪声敏感 |
| **完全连接**（Complete） | 两簇中最远的两点距离 | 倾向生成大小均匀的紧凑球形簇 |
| **平均连接**（Average） | 所有跨簇点对的平均距离 | 介于单/完全之间，较稳健 |
| **Ward 连接** | 合并后 WCSS 的增量 | 最小化簇内方差，与 K-means 目标一致 |

**实践推荐**：Ward 连接在大多数情况下效果最好，是默认首选（详见附录 B）。

![连接方式对比](./figs/ml_cluster_fig08_linkage.png){width=100%}

**图 8** 三种连接方式在客户数据集上的聚类结果（均切割为 3 个簇）。单连接产生了明显的链式效应；完全连接结果较均衡；Ward 连接结果最为整洁。

### 从树状图切割：选择簇的数量

切割规则直观：在某个高度画一条水平线，该线穿过几条竖线，就得到几个簇。

选择切割高度的原则：寻找树状图中**最长的一段竖线**（合并之前保持很长时间未被合并），在该竖线的中部切割。竖线越长，说明这个合并步骤跨越的距离越大，意味着合并前的两组差异显著。

![切割高度](./figs/ml_cluster_fig09_cut.png){width=100%}

**图 9** (a) 同一树状图上的两条切割线：高切割（绿色点划线）得到 2 个大类，低切割（橙色虚线）得到 4 个细分类；(b) 4-簇划分在 PCA 前两个主成分上的散点。

---

## K-means vs 层次聚类

图 10 在债券数据集上直接比较两种方法的结果。

![方法对比](./figs/ml_cluster_fig10_compare.png){width=100%}

**图 10** K-means（左）和层次聚类 Ward（右）在债券数据集上的结果（均取 4 个簇）。ARI（调整兰德指数）衡量与真实类别的一致性，越接近 1 越好。两种方法的结果相近，但在少数边界样本上存在差异。

下表从多个维度对比两种方法：

| | K-means | 层次聚类（Ward） |
|---|---|---|
| **需要预设 K？** | 是 | 否（事后从树状图决定） |
| **时间复杂度** | $O(nKd \cdot \text{iter})$，大数据友好 | $O(n^2 \log n)$，样本量受限 |
| **对初始化的依赖** | 高（需多次运行） | 无（确定性算法） |
| **对离群值的敏感性** | 高（均值易被拉偏） | 中等（Ward 连接较稳健） |
| **簇形状假设** | 球形、大小相近 | 相对灵活 |
| **结果可视化** | 散点图 + Voronoi | 树状图（记录完整合并历史） |
| **适合场景** | 大数据、K 已知或用肘部/轮廓确定 | 中小数据、需要层级结构、K 不确定 |

**实践建议**：两种方法经常一起用——先用层次聚类看树状图，得到对 K 的直觉，再用 K-means 在大数据上做最终聚类。

---

## K-means 的局限：非球形簇

除了对离群值敏感，K-means 还有一个根本性的局限：**假设每个簇大致球形且大小相近**。

当真实的簇形状不满足这个假设时，K-means 会系统性地失败。图 11 展示了三个典型的失败案例。

![非球形簇](./figs/ml_cluster_fig11_nonconvex.png){width=100%}

**图 11** K-means 对非球形数据的失败（颜色=K-means 预测结果，形状=真实类别，空心圆圈=错分点）。(a) 月牙形；(b) 同心圆；(c) 交叉细长椭圆。三种情形的错分率均超过 30%。

::: {.callout-note}
### DBSCAN：基于密度的聚类

对于非球形、密度不均匀的数据，**DBSCAN**（Density-Based Spatial Clustering of Applications with Noise）是 K-means 的有力补充：

- **核心思想**：以密度定义簇——某点周围半径 $\varepsilon$ 内至少有  $\text{min\_samples}$ 个点，则该点为核心点；核心点与其邻域内的核心点递归相连
- **优点**：可以发现任意形状的簇，自动将低密度区域的点标为噪声（标签 $= -1$）
- **缺点**：对参数 $\varepsilon$ 和 $\text{min\_samples}$ 敏感，  高维数据中密度概念退化
- **适用场景**：有明显密度差异的数据，或需要显式检测离群点时

```python
from sklearn.cluster import DBSCAN
db = DBSCAN(eps=0.3, min_samples=5)
labels = db.fit_predict(X_sc)
# labels == -1 的点是噪声/离群点
```

DBSCAN 的工作原理动态演示可参考：[Pinecone 可视化教程](https://www.pinecone.io/learn/k-means-clustering/)
:::

::: {.callout-tip}
### 金融应用：市场状态聚类

聚类分析在量化金融中一个重要的应用是**市场状态识别**（Market Regime Detection）。

典型做法是：以每个月的若干市场指标（波动率、收益率、成交量、利差等）构成特征向量，对所有历史时间段做聚类，将市场状态划分为牛市、熊市、低波动震荡等若干类型。

识别当前所处的市场状态后，可以动态调整组合的仓位和对冲策略——这是动态资产配置和风险平价策略的常见技术基础。

注意事项：金融时序数据具有序列相关性，直接应用 K-means（假设观测独立）可能不最优。隐马尔可夫模型（HMM）是一个考虑了状态转移的替代方案。
:::

---

## 高维数据的聚类：先 PCA 降维再聚类

### 维度诅咒与聚类

当特征维度 $p$ 很高时，聚类面临**维度诅咒**：

- 高维空间中所有点对之间的距离趋于相等，欧氏距离的区分能力大幅下降
- 大量特征可能是噪声，与真正的簇结构无关，污染距离计算

一个有效的应对策略是：**先用 PCA 将数据降维，保留主要信息，再做聚类**。

![PCA+聚类](./figs/ml_cluster_fig12_pca_cluster.png){width=100%}

**图 12** 直接在 4 维空间 K-means（左）vs 先 PCA 降到 2 维再 K-means（右）。两种方案的 ARI 相近，但右图的结果在 PC1-PC2 平面上更直观，对应的债券类别也更容易解读。

**PCA 保留几个主成分用于聚类？**一般原则是保留累计方差超过 80%–90% 的最少主成分数。维度诅咒在 $p > 20$ 后才开始显著，$p > 100$ 时几乎必须先降维。

---

## Python 实践

### 数据准备

```python
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

df = pd.read_csv('./data/cluster_customers.csv')
X = df[['annual_spend', 'freq_per_month']].values

# 标准化（聚类对量纲敏感，必须标准化）
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)
```

::: {.callout-important}
### 聚类也必须标准化

K-means 和层次聚类都基于欧氏距离，对特征的量纲极度敏感。

假设年均消费额的范围是 500–10000 元，月均消费频率的范围是 1–15 次，未标准化时消费额对距离的贡献是频率的数百倍，算法实际上只在消费额维度上做聚类，完全忽略频率信息。

**在聚类之前，务必对所有特征做 StandardScaler（或 MinMaxScaler）。**
:::

### K-means 完整流程

```python
# 1. 肘部法则
inertias = []
for k in range(1, 10):
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, random_state=42)
    inertias.append(km.fit(X_sc).inertia_)

# 2. 轮廓系数
sil_scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    sil_scores[k] = silhouette_score(X_sc, km.fit_predict(X_sc))

best_k = max(sil_scores, key=sil_scores.get)
print(f'轮廓系数最优 K = {best_k}')

# 3. 最终拟合
km_final = KMeans(n_clusters=best_k, init='k-means++',
                  n_init=20, random_state=42)
km_final.fit(X_sc)
labels       = km_final.labels_
centers_orig = scaler.inverse_transform(km_final.cluster_centers_)

for k in range(best_k):
    print(f'簇{k}: 年消费={centers_orig[k,0]:.0f}元，'
          f'月频率={centers_orig[k,1]:.1f}次，'
          f'样本数={( labels==k).sum()}')
```

### 层次聚类完整流程

```python
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

df_B = pd.read_csv('./data/cluster_bonds.csv')
X_B  = df_B[['duration','credit_spread','liquidity','rating_score']].values
X_B_sc = StandardScaler().fit_transform(X_B)

# Ward 层次聚类
Z = linkage(X_B_sc, method='ward')

# 树状图
fig, ax = plt.subplots(figsize=(8, 5))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_font_size=0, color_threshold=Z[-3, 2])
ax.axhline((Z[-3,2]+Z[-4,2])/2, color='red', linestyle='--')
plt.tight_layout(); plt.show()

# 切割为 4 个簇
labels_hc = fcluster(Z, t=4, criterion='maxclust') - 1

# 评估（如果有真实标签）
from sklearn.metrics import adjusted_rand_score
y_true = df_B['true_cluster'].values
print(f'ARI = {adjusted_rand_score(y_true, labels_hc):.4f}')
```

### PCA + K-means

```python
from sklearn.decomposition import PCA

scaler_B = StandardScaler()
X_B_sc   = scaler_B.fit_transform(X_B)

# 保留累计方差 90% 所需的主成分数
pca = PCA(n_components=0.90)
X_B_pca = pca.fit_transform(X_B_sc)
print(f'保留主成分数: {pca.n_components_}')
print(f'累计解释方差: {pca.explained_variance_ratio_.sum():.1%}')

km_pca = KMeans(n_clusters=4, n_init=30, random_state=42)
labels_pca = km_pca.fit_predict(X_B_pca)

print(f'轮廓系数: {silhouette_score(X_B_pca, labels_pca):.4f}')
print(f'ARI: {adjusted_rand_score(y_true, labels_pca):.4f}')
```

---

## 小结

$$
\underbrace{\text{K-means}}_{\min\sum_k\sum_{i\in C_k}\|\mathbf{x}_i-\boldsymbol{\mu}_k\|^2}
\xrightarrow{\text{选 K}}
\underbrace{\text{肘部 + 轮廓系数}}_{\text{统计准则}}
+
\underbrace{\text{业务解读}}_{\text{最终决策}}
$$

**七个关键结论：**

1. **好的聚类**需要同时满足：簇内距离小（intracluster，紧凑）+ 簇间距离大（intercluster，分离）

2. **聚类必须标准化**：欧氏距离对量纲敏感，未标准化的特征会主导聚类结果

3. **K-means 是局部最优算法**：使用 `init='k-means++'` 和 `n_init≥10`，   取多次运行中 WCSS 最小的结果

4. **K-means 对离群值敏感**：极端离群点会将簇中心拉偏，   聚类前应先做异常值检测

5. **肘部法则 + 轮廓系数联合使用**：统计准则给出候选 K，   最终 K 的选择还需结合业务含义

6. **层次聚类不需要预设 K**：树状图提供数据层级结构的完整视图；   Ward 连接是首选，与 K-means 目标函数一致

7. **高维数据先 PCA 降维再聚类**：去除噪声维度，提升距离的区分能力

---

## 附录 A　K-means 收敛性证明

**命题**：K-means 的两步迭代算法在有限步内收敛。

**证明思路**：

**E 步不增加目标函数**：给定中心 $\{\boldsymbol{\mu}_k^{(t)}\}$，将每个点分配到最近的中心，对每个 $i$ 独立取最小值，因此 $\mathcal{L}^{(t+1,\text{E})} \leq \mathcal{L}^{(t)}$。

**M 步不增加目标函数**：给定划分，各簇内最小化 $\sum_{i \in C_k}\|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$的最优解为簇均值，因此 $\mathcal{L}^{(t+1)} \leq \mathcal{L}^{(t+1,\text{E})} \leq \mathcal{L}^{(t)}$。

**有限步内收敛**：$n$ 个点划分为 $K$ 个非空簇共有 $K^n$ 种有限划分方案，$\mathcal{L}$ 单调不增，因此算法有限步内必然停止于某个局部最优解。$\square$

## 附录 B　Ward 连接的数学形式

Ward 连接定义两个簇 $A$ 和 $B$ 的距离为合并后的 WCSS 增量：

$$
d_{\text{Ward}}(A, B)
= \frac{|A| \cdot |B|}{|A| + |B|} \cdot
\|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|^2 \tag{B.1}
$$

公式 (B.1) 说明：Ward 距离正比于两个簇均值的平方距离，以簇大小的调和均值加权。合并大小相近、均值接近的两个簇代价最小。

Ward 连接在每步选择使 WCSS 增量最小的合并，与 K-means 最小化总 WCSS 的目标完全一致，因此两者在最终结果上往往非常接近。

## 参考文献

- James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An Introduction to Statistical Learning: with Applications in Python* (2nd ed.). Springer. Chapter 12. [Link](https://www.statlearning.com/)
- Lloyd, S. P. (1982). Least squares quantization in PCM. *IEEE Transactions on Information Theory*, 28(2), 129–137. [Link](https://doi.org/10.1109/TIT.1982.1056489)
- Arthur, D., & Vassilvitskii, S. (2007). K-means++: The advantages of careful seeding. *Proceedings of the 18th Annual ACM-SIAM Symposium on Discrete Algorithms*, 1027–1035. [Link](https://dl.acm.org/doi/10.5555/1283383.1283494)
- Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *Journal of the American Statistical Association*, 58(301), 236–244. [Link](https://doi.org/10.1080/01621459.1963.10500845)
- Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD*, 96(34), 226–231.

**网络资源**

- Pinecone. K-Means Clustering Tutorial. [https://www.pinecone.io/learn/k-means-clustering/](https://www.pinecone.io/learn/k-means-clustering/)
- Blopig. K-Means Clustering Made Simple. [https://www.blopig.com/blog/2020/07/k-means-clustering-made-simple/](https://www.blopig.com/blog/2020/07/k-means-clustering-made-simple/)